# 16S rRNA refseq name checker
## Download all 16S rRNA refseq records and check the organism names for changes and updates needed

In [3]:
import Bio
import pandas as pd 
import os
import sys
import numpy as np
from Bio import SeqIO
pd.set_option('display.max_rows', 500)
pd.set_option('display.width', 2000)

In [2]:
pip install bio

export: Command not found.
Looking in indexes: https://artifactory.ncbi.nlm.nih.gov/artifactory/api/pypi/python-virtual-repo/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 20.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [bio]5/6 [bio]ne]gs-client]

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
os.system("esearch -db nuccore -query \"33175[BioProject] OR 33317[BioProject]\" | efetch -format gb > refseq4.gbk ")

0

In [5]:
inputfile = (r'refseq4.gbk')
feature_count = 0
orgname = []
defline = []
table = []
report = []
for seq_record in SeqIO.parse(inputfile, "genbank"): 
    dir(seq_record)
    str_id = seq_record.id
    orgname = seq_record.annotations["organism"]
    defline = seq_record.description
    table = str_id, orgname, defline
    report.append(table)

                    



In [6]:
report_df = pd.DataFrame(report, columns=["accession", "orgname", "defline"])
report_df = report_df.set_index('accession')
report_df

,orgname,defline
accession,,
NR_024570.1,Escherichia coli,Escherichia coli strain U 5/41 16S ribosomal R...
NR_024571.1,Brachybacterium muris,Brachybacterium muris strain C3H-21 16S riboso...
NR_025630.1,Streptococcus marimammalium,Streptococcus marimammalium strain M54/01/1 16...
NR_025631.1,Methylobacterium aquaticum,Methylobacterium aquaticum strain GR16 16S rib...
NR_025632.1,Methylobacterium hispanicum,Methylobacterium hispanicum strain GP34 16S ri...
...,...,...
NR_201928.1,Ruicaihuangia ninglii,Ruicaihuangia ninglii strain LJ205 16S ribosom...
NR_201929.1,Sphingobium erigeronicola,Sphingobium erigeronicola strain HWE2-09 16S r...
NR_201930.1,Sphingobium wonjuense,Sphingobium wonjuense strain CR2-8 16S ribosom...


In [7]:
#print(report_df.apply(lambda x: ' '.join([i for i in x['orgname'].split() if i not in x['defline'].split()]), axis=1))
differences_df = report_df.apply(lambda x: ' '.join([i for i in x['orgname'].split() if i not in x['defline'].split()]), axis=1)
differences_df.columns=["accession", "newname"]
#differences_df
#print("Dataframe columns:", differences_df.columns)

In [13]:
differences_df = differences_df.replace(r'^\s*$', np.nan, regex=True)
differences_df

accession
NR_024570.1    NaN
NR_024571.1    NaN
NR_025630.1    NaN
NR_025631.1    NaN
NR_025632.1    NaN
              ... 
NR_201928.1    NaN
NR_201929.1    NaN
NR_201930.1    NaN
NR_201931.1    NaN
NR_201932.1    NaN
Length: 28037, dtype: str

## Generate a report of accessions that have a name change

In [14]:
differences_df.dropna(inplace=True)
differences_df

accession
NR_036912.1                             Pseudoselenomonas
NR_041139.1                                    nigrescens
NR_041224.1                                     violaceus
NR_175410.1                   Anaerocellum diazotrophicum
NR_175415.1                                Cognatitamlana
NR_175418.1                            Alterirhizorhabdus
NR_175437.1                            Allosphingorhabdus
NR_175517.1                        (ex Zhang et al. 2020)
NR_175560.1                   Anaerocellum diazotrophicum
NR_175561.1                   Anaerocellum diazotrophicum
NR_175566.1                            Allosphingorhabdus
NR_176478.1                                 Pseudanabaena
NR_176515.1                          Lyngbya confervoides
NR_176576.1                                       Lyngbya
NR_177675.1                                      tharense
NR_177676.1                                      tharense
NR_177874.1                         Leptolyngbya alkalina
NR_1

In [15]:
differences_df.shape


(140,)

In [16]:
differences_df

accession
NR_036912.1                             Pseudoselenomonas
NR_041139.1                                    nigrescens
NR_041224.1                                     violaceus
NR_175410.1                   Anaerocellum diazotrophicum
NR_175415.1                                Cognatitamlana
NR_175418.1                            Alterirhizorhabdus
NR_175437.1                            Allosphingorhabdus
NR_175517.1                        (ex Zhang et al. 2020)
NR_175560.1                   Anaerocellum diazotrophicum
NR_175561.1                   Anaerocellum diazotrophicum
NR_175566.1                            Allosphingorhabdus
NR_176478.1                                 Pseudanabaena
NR_176515.1                          Lyngbya confervoides
NR_176576.1                                       Lyngbya
NR_177675.1                                      tharense
NR_177676.1                                      tharense
NR_177874.1                         Leptolyngbya alkalina
NR_1

## Output list of accessions with name change to a file 

In [17]:
#differences_df.to_csv('differences_df', index=False, header=False, sep ='\t', columns=['accession', 'orgname', 'defline'])
differences_df.to_csv('differences_df', index=True, header=False, sep ='\t')
#save differences acclist as a file
differences_acc_df = pd.DataFrame(index=differences_df.index)
differences_acc_df.to_csv('NRacclist.csv', index=True, header=False)

In [11]:
#seqfetch
#using a test file here real file is NRacclist.csv
os.system("seqfetch -G NRacclist_b.csv -t 1 > NRacc.asn")
#se2bss
os.system("se2bss -i NRacc.asn -o NRacc.pbs")
#asn_cleanup
os.system("asn_cleanup -i NRacc.pbs -X r -o NRacc.sqn -T -logfile log ")

0

In [12]:
#edit asnfile to to convert to bioseq
def replace_lines(input_file, output_file, replacements):
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        for line in infile:
            for old, new in replacements.items():
                line = line.replace(old, new)
            outfile.write(line)

replacements = {
    "Seq-entry ::= set {": "Bioseq-set ::= {",
    "  class genbank,\n": ""
}

replace_lines("NRacc.sqn", "NRacc.bioseq", replacements)

In [6]:
password = (r'pwd')
#idload these steps aren't working yet
#os.system("source \/am\/ncbiapdata\/database_env\/id_env.csh")
os.system("source /am/ncbiapdata/database_env/id_env.csh")
os.system("setenv IDLOADSEQENTRY \"idloadseqentry -O AllowGiFarPtr,AllowSought,AllowAlignFarPtr,UseIdDates\"")
os.system("idload.pl -a NRacc.bioseq -d `pwd` -l `pwd`/idload -P till1620 -o NCBI-LegRefSeq")
#confirm successful load
#send message with log files to user reporting loads and log files

/am/ncbiapdata/database_env/id_env.csh: line 26: setenv: command not found
/am/ncbiapdata/database_env/id_env.csh: line 40: setenv: command not found
/am/ncbiapdata/database_env/id_env.csh: line 57: setenv: command not found
/am/ncbiapdata/database_env/id_env.csh: line 58: setenv: command not found
/am/ncbiapdata/database_env/id_env.csh: line 68: setenv: command not found
/am/ncbiapdata/database_env/id_env.csh: line 69: setenv: command not found
/am/ncbiapdata/database_env/id_env.csh: line 78: setenv: command not found
sh: setenv: command not found
/net/idflow02/export/d0/gbutils/idload.pl : ID_OPEN_SERVER environment variable not defined


65280

In [7]:
os.system("idstat -a U12345")


ID Database Sequence Loading History

Accession          GI         SatKey     SatName   State Owner                 Wdrw Supp Date Loaded           Username
------------------ ---------- ---------- --------- ----- --------------------- ---- ---- --------------------- --------
U12345             540023     19116189   NCBI      Alive SUB-update            No   No   9/21/2007 4:38:44PM   yankie
U12345             540023     88495743   OLD12     Dead  SUB-update            No   No   5/7/1999 4:24:10PM    kim
U12345             540023     41647313   OLD12     Dead  SUB-update            No   No   3/2/1999 3:34:23PM    jacques
U12345             540023     41351081   OLD12     Dead  SUB-update            No   No   1/27/1999 1:00:15PM   jacques
U12345             540023     40134808   OLD12     Dead  SUB-update            No   No   10/8/1994 1:21:43AM   cavanaug
U12345             540023     40114900   OLD12     Dead  NCBI-submissions      No   No   10/1/1994 10:14:44AM  cavanaug
U12345    

0